## Coding (with Agents)

There are many ways to build reproducible software environments for Python and many tools to help write Python code.  Here we will discuss using Conda for creating environments, and JupyterLab, VSCode and Gemini for creating code. 

* Python, Pip and Conda, Conda environment files 
* Jupyterlab IDE 
* Claude Code 
* VSCode IDE with Claude Code
* Gemini CLI 

## Installing Python
We are using Coder for this workshop, but if you want to install Python on your computer, [Micromamba](https://mamba.readthedocs.io/en/latest/installation/micromamba-installation.html) is generally considered an excellent and safe choice for installing Python and managing environments without the licensing concerns associated with the default channels in commercial Python distributions like Anaconda or standard Miniconda.

## Creating Python Environments
Using [conda environment files](https://docs.conda.io/projects/conda/en/latest/user-guide/tasks/manage-environments.html#creating-an-environment-from-an-environment-yml-file) is the best way to create and maintain an environment for general use in cloud-native geoprocessing.   

We used the conda environment files [main](./protocoast-notebook_env.yml) and [develop](./eopf-notebook_env.yml) to create the conda environments used in this course. 

## Coding in the JupyterLab environment
JupyterLab offers a flexible and powerful environment for writing and executing Python code, often serving as a de facto IDE for data science and interactive computing. While not a traditional IDE like PyCharm or VS Code, its interface, composed of integrated notebooks, consoles, terminals, and a file browser, allows for a unique, highly iterative workflow. Users can write, test, and document code seamlessly within the same notebook document, executing cells individually and viewing output, plots, or dataframe previews immediately below the code. This blend of interactive computing, code editing, and rich output visualization makes JupyterLab exceptionally well-suited for exploratory data analysis, machine learning prototyping, and developing educational materials.

## Coding in VSCode 
Visual Studio Code (VS Code) has emerged as an exceptionally popular and versatile Integrated Development Environment (IDE) for Python, largely thanks to its extensive set of extensions. The core strength of VS Code for Python development lies in its ability to offer a lightweight yet full-featured experience, including sophisticated features like intelligent code completion (IntelliSense), debugging with breakpoints, and integrated version control via Git. Furthermore, it supports features essential for modern Python workflows, such as managing virtual environments, running test frameworks, and most importantly, integrating directly with Jupyter notebooks and interactive Python consoles. This seamless integration and customization via extensions make it an efficient and comfortable environment for projects ranging from small scripts to large-scale applications.

## Coding with LLM assistants (ChatGPT, Claude, Grok, Gemini)
The power of using the LLMs like Gemini Code Assist extension in VS Code fundamentally shifts the development experience from reactive coding to proactive AI assistance. Integrated directly into the editor, Gemini provides instant, context-aware code completions as you type, and can generate entire functions or code blocks from simple natural language comments. Beyond generation, its true power lies in its transformation capabilities: developers can leverage "smart actions" and commands like /fix, /doc, and /simplify to automatically debug errors, generate comprehensive unit tests, or refactor complex code, all within a quick, integrated flow. This on-demand, deep understanding of the local codebase, combined with a conversational chat assistant, dramatically accelerates the development lifecycle, allowing programmers to minimize time spent on boilerplate and debugging and maximize focus on complex problem-solving.

## Contexts 
`CLAUDE.md` files are always read, not sometimes. The loading is deterministic:

* `~/.claude/CLAUDE.md`: always loaded, every conversation
* `<project>/CLAUDE.md`: always loaded when working in that project
* Parent directory `CLAUDE.md` files: also always loaded (Claude walks up the directory tree)
* `MEMORY.md` index is always in context
* Individual memory files (like `feedback_xr_concat.md`) only get read when Claude explicitly fetch them, 
which is what Claude is supposed to do when they seem relevant, but can miss.

`MEMORY.md` is just an index: a list of one-line pointers. It's always in context so I can see what memories exist, but the actual content lives in the individual files and I only read those on demand.

## Example Failure
* I see the MEMORY.md entry for "Windows paths in WSL"
* I don't judge it relevant to your screenshot request
* I never read `user_windows_paths.md`
* I try the path as-is and fail

Therefore `CLAUDE.md` is more reliable than memory for things you always want enforced.

## How Agents Work

An AI **agent** is a model that doesn't just answer a single question — it takes a sequence of actions to accomplish a goal. Instead of one prompt → one response, the agent runs in a loop:

1. **Observe** — read the current state (files, terminal output, tool results)
2. **Plan** — decide what action to take next
3. **Act** — call a tool (run a command, read a file, edit code, search the web)
4. **Repeat** — incorporate the result and decide the next step

This loop continues until the agent decides the goal is complete, or the user interrupts.

### The Context Window

The context window is the agent's working memory — everything it can "see" at once: the conversation history, the system prompt, tool results, and any files it has read. Think of it as a finite whiteboard.

* Current frontier models have context windows of roughly **100,000–1,000,000 tokens** (a token ≈ ¾ of a word)
* The entire conversation — your messages, the agent's replies, and tool results — accumulates inside this window
* When the window fills, older content is **compressed or dropped** automatically
* Larger context = slower and more expensive inference

### Claude Code as a Concrete Example

[Claude Code](https://claude.ai/claude-code) is a CLI agent built on Claude. When you start a session:

* The system prompt describes the tools available (Read, Edit, Bash, Grep, Glob, WebSearch, …)
* Your CLAUDE.md files are injected at the start
* Each turn you type is appended to the context
* When Claude calls a tool (e.g. runs `git diff`), the output is appended to the context as a tool result
* Claude then decides whether to call another tool or respond to you

Claude Code has access to tools including: reading/writing files, running shell commands, searching the web, spawning sub-agents, and reading/writing memory files. The agent decides which tools to call and in what order — you provide the goal, the agent figures out the steps.

## Memory in Claude Code

Claude Code has several layers of memory, each with different scope and persistence:

**1. In-context memory (conversation history)**
The active conversation — everything said in the current session. Fast and immediately accessible, but finite. When the context window fills up, older messages are compressed or dropped. Nothing here survives after the session ends.

**2. CLAUDE.md — persistent instructions**
Markdown files that are *always* loaded into every conversation. They are deterministic: if the file exists, it is read.

* `~/.claude/CLAUDE.md` — global, loaded for every project
* `<project>/CLAUDE.md` — project-specific, loaded whenever Claude Code runs in that directory
* Parent directories are also scanned up the tree

Use CLAUDE.md for things you always want enforced: coding conventions, environment notes, workflow rules. This is the most reliable form of persistent context.

**3. Memory files — recalled on demand**
`MEMORY.md` is a short index file always in context. It lists pointers to individual memory files (e.g. `feedback_testing.md`, `project_data_access.md`). Claude reads the index automatically, but only fetches individual files when they seem relevant — so critical information is better placed in CLAUDE.md than in a memory file.

**4. Tool results (ephemeral)**
File reads, shell output, web fetches — these appear in the context window for the duration of the conversation but are not persisted anywhere automatically.

### Summary

| Layer | Always loaded? | Survives session? | Best for |
|-------|---------------|-------------------|----------|
| Conversation history | Yes (active session) | No | Working context |
| CLAUDE.md | Yes (deterministic) | Yes | Rules, conventions, always-on facts |
| Memory files | Index yes; content on-demand | Yes | Preferences, project history, user notes |
| Tool results | While in context | No | Fetched data, command output |

## Common Things to Be Aware Of

**Context window limits and compaction**
Every conversation has a finite context window. As a session grows long, older messages are summarized or dropped. Important details established early in a conversation may "fall out" of context. For long tasks, periodically recap key facts or re-state constraints.

**Silent wrong answers**
Unlike a syntax error, a wrong agent output often looks completely reasonable. Agents can confidently produce plausible-but-incorrect code, analysis, or explanations. Always verify outputs against ground truth, especially for scientific or safety-critical work.

**Prompt injection**
If the agent reads external content (files, web pages, tool results), that content could contain instructions designed to hijack the agent's behavior. Review tool results if something seems off.

**Cost and token usage**
Every message costs tokens. Long context windows, large file reads, and many tool calls add up. Estimate costs before running long autonomous loops, and prefer targeted reads over reading entire files.

**Destructive operations**
Agents can run shell commands, delete files, push to git remotes, and send messages. By default Claude Code asks for confirmation before risky irreversible actions, but it's worth understanding your permission settings and reviewing what the agent is about to do.

**Determinism and reproducibility**
The same prompt does not guarantee the same output. For reproducible scientific workflows, capture the exact code and data the agent produced, not just the prompt that asked for it.

**Model drift between sessions**
An agent has no memory of previous sessions unless you provide it explicitly (via CLAUDE.md or memory files). Preferences, decisions, and context from a previous conversation are lost unless persisted.

## Agentic Engineering vs Traditional Coding

Working with AI agents is a genuinely different discipline from writing code directly. The table below highlights key differences:

| Aspect | Traditional Coding | Agentic Engineering |
|--------|-------------------|---------------------|
| **Primary input** | Source code you write | Natural language prompts |
| **Iteration cycle** | Write → run → debug | Prompt → review → refine |
| **Error mode** | Explicit crash or error message | Silent, plausible-but-wrong output |
| **Reproducibility** | Deterministic | Probabilistic — same prompt may yield different results |
| **State** | Variables, call stack | Context window + persistent files |
| **Testing** | Unit / integration tests | Output review, spot-checking, human evaluation |
| **Version control** | Code diffs (meaningful) | Prompt + output diffs (harder to audit) |
| **Key skill** | Syntax, algorithms, architecture | Clear decomposition, precise prompting, verification |
| **Scaling** | Linear with code complexity | Requires task decomposition for complex goals |
| **Cost model** | CPU/memory/time | Token usage (input + output) per request |

The most important shift: **you are now a reviewer, not a typer**. Your job is to write clear instructions, critically evaluate what the agent produces, and course-correct — not to produce every line yourself.